# Run the photometric robustness benchmark from a YAML configuration

This notebook reproduces the final benchmark pipeline in a reviewer-friendly, plug-and-play format.

It runs:

- paired clean/adverse evaluation on the synthetic dataset
- masked pixel agreement
- class-wise retention by condition
- grouped barplots for the three paper scenarios
- qualitative SegFormer worst-case visualizations

Configuration is loaded from a YAML file so that paths, thresholds, and model choices are easy to modify.


In [ ]:
from pathlib import Path

# ==== USER CONFIG ====
CONFIG_PATH = Path("./benchmark_config.yaml")
assert CONFIG_PATH.exists(), f"Config file not found: {CONFIG_PATH}"
print("Using config:", CONFIG_PATH.resolve())


In [ ]:
import os
import json
import random
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import yaml
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
})

from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor


In [ ]:
with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

cfg


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.get("seed", 0))

DATA_DIR = cfg["data_dir"]
SAVE_DIR = Path(cfg["save_dir"])
DEVICE = cfg.get("device", "cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = int(cfg.get("batch_size", 2))
NUM_WORKERS = int(cfg.get("num_workers", 4))
CONF_THRESHOLD = float(cfg.get("conf_threshold", 0.6))
MASS_COVERAGE = float(cfg.get("mass_coverage", 0.99))
N_QUAL_SAMPLES = int(cfg.get("n_qual_samples", 6))
QUAL_SEGFORMER_ID = cfg.get("qual_segformer_id", "nvidia/segformer-b5-finetuned-cityscapes-1024-1024")
PLOT_SCENARIOS = list(cfg.get("plot_scenarios", ["Day-RAIN", "Sunset-FOGGY", "Night-RAIN"]))
PLOT_DROP_CLASSNAME = cfg.get("plot_drop_classname", "train")
ENABLE_MASK2FORMER = bool(cfg.get("enable_mask2former", True))

SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("SAVE_DIR:", SAVE_DIR.resolve())


In [ ]:
CITYSCAPES_PALETTE_19 = np.array([
    (128,  64, 128), (244,  35, 232), ( 70,  70,  70), (102, 102, 156),
    (190, 153, 153), (153, 153, 153), (250, 170,  30), (220, 220,   0),
    (107, 142,  35), (152, 251, 152), ( 70, 130, 180), (220,  20,  60),
    (255,   0,   0), (  0,   0, 142), (  0,   0,  70), (  0,  60, 100),
    (  0,  80, 100), (  0,   0, 230), (119,  11,  32),
], dtype=np.uint8)


In [ ]:
class PairedImageDataset(Dataset):
    def __init__(self, data_dir, mode="random", transform=None):
        self.data_dir = data_dir
        self.mode = mode.lower()
        self.transform = transform if transform else transforms.ToTensor()

        self.day_dir = os.path.join(data_dir, "Day")
        self.sunset_dir = os.path.join(data_dir, "Sunset")
        self.night_dir = os.path.join(data_dir, "Night")
        self.image_pairs = self._group_images()

    def _group_images(self):
        all_images = {"Day": {}, "Sunset": {}, "Night": {}}

        def collect(folder, category):
            if not os.path.exists(folder):
                return
            for f in sorted(os.listdir(folder)):
                if f.lower().endswith((".jpg", ".png", ".jpeg")):
                    key = f.split("_")[0]
                    all_images[category].setdefault(key, []).append(os.path.join(folder, f))

        collect(self.day_dir, "Day")
        collect(self.sunset_dir, "Sunset")
        collect(self.night_dir, "Night")

        pairs = []
        for key, imgs in all_images["Day"].items():
            base = [x for x in imgs if "Day_EXTRASUNNY" in os.path.basename(x)]
            if not base:
                continue
            others = all_images["Day"].get(key, []) + all_images["Sunset"].get(key, []) + all_images["Night"].get(key, [])
            for b in base:
                pairs.append((key, b, others))
        return pairs

    def __len__(self):
        return len(self.image_pairs)

    def __getitem__(self, idx):
        loc_id, main_path, others = self.image_pairs[idx]

        if self.mode.startswith("d"):
            cat = "Day"
        elif self.mode.startswith("s"):
            cat = "Sunset"
        elif self.mode.startswith("n"):
            cat = "Night"
        else:
            cat = random.choice(["Day", "Sunset", "Night"])

        weather = None
        if len(self.mode) > 1:
            w = self.mode[1]
            weather = {"f": "FOGGY", "r": "RAIN", "s": "EXTRASUNNY"}.get(w)

        matches = [x for x in others if f"/{cat}/" in x.replace("\\", "/")]
        if weather:
            matches = [x for x in matches if weather in os.path.basename(x)]
        if not matches:
            matches = others

        sec_path = random.choice(matches)
        while sec_path == main_path and len(matches) > 1:
            sec_path = random.choice(matches)

        main = Image.open(main_path).convert("RGB")
        sec = Image.open(sec_path).convert("RGB")

        main_t = self.transform(main)
        sec_t = self.transform(sec)

        b = os.path.basename(sec_path)
        parts = b.split("_")
        sec_illum = parts[1] if len(parts) > 1 else "UNK"
        sec_weather = parts[2].split(".")[0] if len(parts) > 2 else "UNK"
        condition = f"{sec_illum}-{sec_weather}"

        return {
            "pair": torch.stack([main_t, sec_t], dim=0),
            "loc_id": loc_id,
            "sec_condition": condition,
            "clean_path": main_path,
            "sec_path": sec_path,
        }


In [ ]:
def pixel_agreement_masked(pred_a: torch.Tensor, pred_b: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    same = (pred_a == pred_b) & mask
    denom = mask.flatten(1).sum(dim=1).clamp_min(1)
    num = same.flatten(1).sum(dim=1)
    return (num.double() / denom.double())

def batch_counts_by_condition(pred_clean, pred_adv, conf_mask, conditions, num_classes: int):
    idx_by_cond = defaultdict(list)
    for i, c in enumerate(conditions):
        idx_by_cond[c].append(i)

    out = {}
    for c, idxs in idx_by_cond.items():
        idxs_t = torch.tensor(idxs, device=pred_clean.device, dtype=torch.long)

        a = pred_clean.index_select(0, idxs_t)
        b = pred_adv.index_select(0, idxs_t)
        m = conf_mask.index_select(0, idxs_t)

        a_f = a[m]
        b_f = b[m]

        valid = (a_f >= 0) & (a_f < num_classes) & (b_f >= 0) & (b_f < num_classes)
        a_f = a_f[valid]
        b_f = b_f[valid]

        mass = torch.bincount(a_f, minlength=num_classes).to(torch.float64)
        same = torch.bincount(a_f[a_f == b_f], minlength=num_classes).to(torch.float64)
        out[c] = (same.detach().to("cpu"), mass.detach().to("cpu"))
    return out

def select_classes_by_mass_coverage(df_s: pd.DataFrame, coverage: float, drop_class: str = None):
    df = df_s.copy()
    if drop_class is not None:
        df = df[df["class_name"] != drop_class]

    mass_by_class = df.groupby("class_name")["mass_clean_masked"].sum().sort_values(ascending=False)
    total = float(mass_by_class.sum())
    if total < 1e-9:
        return []

    cum = mass_by_class.cumsum() / total
    keep = mass_by_class.index[cum <= coverage].tolist()

    if len(keep) == 0:
        keep = [mass_by_class.index[0]]
    elif cum.iloc[len(keep) - 1] < coverage and len(keep) < len(mass_by_class):
        keep.append(mass_by_class.index[len(keep)])

    return keep

def decode_cityscapes_mask(mask_hw: np.ndarray):
    h, w = mask_hw.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)
    for k in range(19):
        out[mask_hw == k] = CITYSCAPES_PALETTE_19[k]
    return out

def tensor_to_uint8_img(x: torch.Tensor):
    return (x.detach().cpu().clamp(0, 1).numpy().transpose(1, 2, 0) * 255.0).astype(np.uint8)

def select_worst_cases(csv_path: str, model_name: str, scenario: str, k: int):
    df = pd.read_csv(csv_path)
    df = df[(df["model"] == model_name) & (df["sec_condition"] == scenario)].copy()
    if df.empty:
        return df

    if "coverage_masked" in df.columns:
        df = df[df["coverage_masked"] >= 0.2]
        if df.empty:
            return df

    df = df.sort_values("pixel_agreement_masked", ascending=True)
    df = df.drop_duplicates(subset=["loc_id"], keep="first")
    return df.head(k)


In [ ]:
class BaseAdapter:
    def __init__(self, name: str):
        self.name = name

    def num_classes(self) -> int:
        raise NotImplementedError

    def id2label(self) -> Optional[Dict[int, str]]:
        return None

    @torch.no_grad()
    def predict(self, images_01: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        raise NotImplementedError


class SegFormerAdapter(BaseAdapter):
    def __init__(self, name: str, model_id: str, device: torch.device):
        super().__init__(name)
        self.device = device
        self.processor = SegformerImageProcessor.from_pretrained(model_id, use_fast=True)
        self.model = SegformerForSemanticSegmentation.from_pretrained(model_id).to(device).eval()
        self._num = int(self.model.config.num_labels)
        self._id2label = dict(self.model.config.id2label)

    def num_classes(self) -> int:
        return self._num

    def id2label(self):
        return self._id2label

    @torch.no_grad()
    def predict(self, images_01: torch.Tensor):
        inputs = self.processor(images=images_01, return_tensors="pt", do_rescale=False)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        out = self.model(**inputs)
        logits = out.logits
        B, _, H, W = images_01.shape
        logits = F.interpolate(logits, size=(H, W), mode="bilinear", align_corners=False)
        probs = F.softmax(logits, dim=1)
        conf = probs.max(dim=1).values
        pred = probs.argmax(dim=1).to(torch.int64)
        return pred, conf

def try_build_mask2former(device: torch.device) -> List[BaseAdapter]:
    adapters = []
    if not ENABLE_MASK2FORMER:
        return adapters
    try:
        from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor
        import scipy  # noqa: F401
    except Exception as e:
        print(f"[warn] Skipping Mask2Former: {e}")
        return adapters

    class Mask2FormerAdapter(BaseAdapter):
        def __init__(self, name: str, model_id: str, device: torch.device):
            super().__init__(name)
            self.device = device
            self.processor = Mask2FormerImageProcessor.from_pretrained(model_id, use_fast=True)
            self.model = Mask2FormerForUniversalSegmentation.from_pretrained(model_id).to(device).eval()
            self._num = int(self.model.config.num_labels)
            self._id2label = dict(self.model.config.id2label)

        def num_classes(self) -> int:
            return self._num

        def id2label(self):
            return self._id2label

        @torch.no_grad()
        def predict(self, images_01: torch.Tensor):
            B, _, H, W = images_01.shape
            images_cpu = [img.detach().cpu() for img in images_01]
            inputs = self.processor(images=images_cpu, return_tensors="pt", do_rescale=False)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            outputs = self.model(**inputs)
            class_logits = outputs.class_queries_logits
            mask_logits = outputs.masks_queries_logits

            class_probs = F.softmax(class_logits, dim=-1)[..., :-1]
            mask_probs = torch.sigmoid(mask_logits)
            mask_probs = F.interpolate(mask_probs, size=(H, W), mode="bilinear", align_corners=False)

            sem_scores = torch.einsum("bqc,bqhw->bchw", class_probs, mask_probs)
            probs = sem_scores / sem_scores.sum(dim=1, keepdim=True).clamp_min(1e-6)

            conf = probs.max(dim=1).values
            pred = probs.argmax(dim=1).to(torch.int64)
            return pred, conf

    adapters.append(Mask2FormerAdapter("Mask2Former-Swin-S", "facebook/mask2former-swin-small-cityscapes-semantic", device))
    adapters.append(Mask2FormerAdapter("Mask2Former-Swin-L", "facebook/mask2former-swin-large-cityscapes-semantic", device))
    return adapters


In [ ]:
@torch.no_grad()
def run_benchmark():
    device = torch.device(DEVICE)

    adapters: List[BaseAdapter] = []
    if cfg.get("models", {}).get("segformer_b0", True):
        adapters.append(SegFormerAdapter("SegFormer-B0", "nvidia/segformer-b0-finetuned-cityscapes-1024-1024", device))
    if cfg.get("models", {}).get("segformer_b2", True):
        adapters.append(SegFormerAdapter("SegFormer-B2", "nvidia/segformer-b2-finetuned-cityscapes-1024-1024", device))
    if cfg.get("models", {}).get("segformer_b5", True):
        adapters.append(SegFormerAdapter("SegFormer-B5", "nvidia/segformer-b5-finetuned-cityscapes-1024-1024", device))
    if cfg.get("models", {}).get("mask2former", True):
        adapters += try_build_mask2former(device)

    if not adapters:
        raise RuntimeError("No models enabled in the YAML configuration.")

    id2label = adapters[0].id2label() or {i: str(i) for i in range(adapters[0].num_classes())}
    K = len(id2label)
    class_names = [id2label[i] for i in range(K)]

    ds = PairedImageDataset(DATA_DIR, mode="random", transform=transforms.ToTensor())
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    rows_img = []
    cond_sum = {a.name: defaultdict(float) for a in adapters}
    cond_n = {a.name: defaultdict(int) for a in adapters}
    cond_same = {a.name: defaultdict(lambda: torch.zeros(K, dtype=torch.float64)) for a in adapters}
    cond_mass = {a.name: defaultdict(lambda: torch.zeros(K, dtype=torch.float64)) for a in adapters}
    global_same = {a.name: torch.zeros(K, dtype=torch.float64) for a in adapters}
    global_mass = {a.name: torch.zeros(K, dtype=torch.float64) for a in adapters}

    for batch in tqdm(loader, desc="Evaluating"):
        pair = batch["pair"]
        clean = pair[:, 0].to(device, non_blocking=True)
        adv = pair[:, 1].to(device, non_blocking=True)

        loc_ids = batch["loc_id"]
        conds = [c if c is not None else "UNKNOWN" for c in batch["sec_condition"]]

        for adp in adapters:
            pred_clean, conf_clean = adp.predict(clean)
            pred_adv, _ = adp.predict(adv)

            conf_mask = conf_clean >= CONF_THRESHOLD
            agree = pixel_agreement_masked(pred_clean, pred_adv, conf_mask)

            counts = batch_counts_by_condition(pred_clean, pred_adv, conf_mask, conds, num_classes=K)
            for c, (same_c, mass_c) in counts.items():
                cond_same[adp.name][c] += same_c
                cond_mass[adp.name][c] += mass_c
                global_same[adp.name] += same_c
                global_mass[adp.name] += mass_c

            for i in range(clean.shape[0]):
                c = conds[i]
                v = float(agree[i].item())
                cond_sum[adp.name][c] += v
                cond_n[adp.name][c] += 1
                coverage = float(conf_mask[i].float().mean().item())

                rows_img.append({
                    "model": adp.name,
                    "loc_id": loc_ids[i],
                    "sec_condition": c,
                    "pixel_agreement_masked": v,
                    "conf_threshold": CONF_THRESHOLD,
                    "coverage_masked": coverage,
                    "clean_path": batch["clean_path"][i],
                    "sec_path": batch["sec_path"][i],
                })

    df_img = pd.DataFrame(rows_img)
    df_img.to_csv(SAVE_DIR / "results_pixel_agreement_per_image.csv", index=False)

    rows_cond = []
    for m in cond_sum:
        for c in cond_sum[m]:
            n = cond_n[m][c]
            rows_cond.append({
                "model": m,
                "sec_condition": c,
                "pixel_agreement_masked_mean": cond_sum[m][c] / max(n, 1),
                "n_pairs": int(n),
                "conf_threshold": CONF_THRESHOLD,
            })
    df_cond = pd.DataFrame(rows_cond)
    df_cond.to_csv(SAVE_DIR / "results_pixel_agreement_by_condition.csv", index=False)

    rows_cls_cond = []
    for m in cond_same:
        for c in cond_same[m]:
            mass = cond_mass[m][c].clamp_min(1.0)
            ret = (cond_same[m][c] / mass)
            for k in range(K):
                rows_cls_cond.append({
                    "model": m,
                    "sec_condition": c,
                    "class_id": k,
                    "class_name": class_names[k],
                    "retention_masked": float(ret[k].item()),
                    "mass_clean_masked": float(cond_mass[m][c][k].item()),
                    "conf_threshold": CONF_THRESHOLD,
                })
    df_cls_cond = pd.DataFrame(rows_cls_cond)
    df_cls_cond.to_csv(SAVE_DIR / "results_class_retention_by_condition.csv", index=False)

    rows_cls_global = []
    for m in global_same:
        mass = global_mass[m].clamp_min(1.0)
        ret = global_same[m] / mass
        for k in range(K):
            rows_cls_global.append({
                "model": m,
                "class_id": k,
                "class_name": class_names[k],
                "retention_masked_global": float(ret[k].item()),
                "mass_clean_masked_global": float(global_mass[m][k].item()),
                "conf_threshold": CONF_THRESHOLD,
            })
    df_cls_global = pd.DataFrame(rows_cls_global)
    df_cls_global.to_csv(SAVE_DIR / "results_class_retention_global.csv", index=False)

    return adapters, df_img, df_cond, df_cls_cond, df_cls_global


In [ ]:
adapters, df_img, df_cond, df_cls_cond, df_cls_global = run_benchmark()
print("Saved benchmark outputs to:", SAVE_DIR.resolve())
df_cond.head()


In [ ]:
plot_dir = SAVE_DIR / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

for scen in PLOT_SCENARIOS:
    df_s = df_cls_cond[df_cls_cond["sec_condition"] == scen].copy()
    if df_s.empty:
        print(f"[warn] Scenario {scen} not found.")
        continue

    keep_classes = select_classes_by_mass_coverage(df_s, coverage=MASS_COVERAGE, drop_class=PLOT_DROP_CLASSNAME)
    if not keep_classes:
        print(f"[warn] No classes selected for {scen}.")
        continue

    df_s = df_s[df_s["class_name"].isin(keep_classes)].copy()
    piv = df_s.pivot_table(index="class_name", columns="model", values="retention_masked").reindex(keep_classes)

   # Grouped barplot
    fig = plt.figure(figsize=(13, 5.5))
    ax = fig.add_subplot(111)
    
    models = list(piv.columns)
    classes = list(piv.index)
    
    x = np.arange(len(classes))
    n_models = len(models)
    bar_width = 0.8 / max(n_models, 1)
    
    for j, m in enumerate(models):
        y = piv[m].values
        ax.bar(
            x + j * bar_width - (n_models - 1) * bar_width / 2,
            y,
            width=bar_width,
            label=m
        )
    
    # Bigger and cleaner fonts
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=12)
    ax.set_ylabel("Retention", fontsize=14)
    ax.tick_params(axis="y", labelsize=12)
    
    # Remove title
    # ax.set_title(...)
    
    # Legend outside
    ax.legend(
        fontsize=11,
        ncol=1,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False
    )
    
    # Optional: nicer y-limits
    ax.set_ylim(0.0, 1.0)
    
    # Leave space on the right for the legend
    fig.tight_layout(rect=[0, 0, 0.82, 1])
    
    # Save as PDF instead of PNG
    fig.savefig(os.path.join(plot_dir, f"plot_bar_class_retention__{scen}.pdf"), bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)


In [ ]:
qual_dir = SAVE_DIR / "qualitative_segformer"
qual_dir.mkdir(parents=True, exist_ok=True)

seg_vis = SegFormerAdapter("SegFormer-QUAL", QUAL_SEGFORMER_ID, torch.device(DEVICE))
csv_img = SAVE_DIR / "results_pixel_agreement_per_image.csv"
model_for_qual = cfg.get("qual_model_name", "SegFormer-B5")

for scen in PLOT_SCENARIOS:
    worst_df = select_worst_cases(csv_img, model_for_qual, scen, k=N_QUAL_SAMPLES)
    if worst_df.empty:
        print(f"[warn] No worst cases found for {model_for_qual} in {scen}")
        continue

    fig = plt.figure(figsize=(14, 3 * len(worst_df)))

    for r, row in enumerate(worst_df.itertuples(index=False)):
        clean_img_pil = Image.open(row.clean_path).convert("RGB")
        adv_img_pil = Image.open(row.sec_path).convert("RGB")

        to_t = transforms.ToTensor()
        clean_t = to_t(clean_img_pil).unsqueeze(0).to(DEVICE)
        adv_t = to_t(adv_img_pil).unsqueeze(0).to(DEVICE)

        pred_c, _ = seg_vis.predict(clean_t)
        pred_a, _ = seg_vis.predict(adv_t)

        clean_img = tensor_to_uint8_img(clean_t[0])
        adv_img = tensor_to_uint8_img(adv_t[0])
        clean_mask = decode_cityscapes_mask(pred_c[0].detach().cpu().numpy().astype(np.int32))
        adv_mask = decode_cityscapes_mask(pred_a[0].detach().cpu().numpy().astype(np.int32))

        ax0 = fig.add_subplot(len(worst_df), 4, r * 4 + 1)
        ax0.imshow(clean_img); ax0.set_title("Clean"); ax0.axis("off")

        ax1 = fig.add_subplot(len(worst_df), 4, r * 4 + 2)
        ax1.imshow(clean_mask); ax1.set_title("Pred clean"); ax1.axis("off")

        ax2 = fig.add_subplot(len(worst_df), 4, r * 4 + 3)
        ax2.imshow(adv_img); ax2.set_title(f"Adverse ({scen})"); ax2.axis("off")

        ax3 = fig.add_subplot(len(worst_df), 4, r * 4 + 4)
        ax3.imshow(adv_mask); ax3.set_title(f"Pred adv\nagree={row.pixel_agreement_masked:.3f}"); ax3.axis("off")

    fig.tight_layout()
    out_path = qual_dir / f"qual_segformer_worst__{scen}.pdf"
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", out_path)


In [ ]:
with open(SAVE_DIR / "run_config_resolved.json", "w") as f:
    json.dump({
        "data_dir": DATA_DIR,
        "save_dir": str(SAVE_DIR),
        "device": DEVICE,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "conf_threshold": CONF_THRESHOLD,
        "mass_coverage": MASS_COVERAGE,
        "plot_scenarios": PLOT_SCENARIOS,
        "drop_class_from_plot": PLOT_DROP_CLASSNAME,
        "models": [a.name for a in adapters],
        "qual_segformer_id": QUAL_SEGFORMER_ID,
        "qual_samples_per_scenario": N_QUAL_SAMPLES,
    }, f, indent=2)

print("Saved:", SAVE_DIR / "run_config_resolved.json")


## Notes for reviewers

- Edit only the YAML file to change dataset path, output directory, thresholds, and enabled models.
- This notebook intentionally mirrors the final benchmark used in the paper.
- DeepLab is not included here; it can be added later once a validated checkpoint is available.
